# Explainer Notebook: Jutland Migration, Gentrification, and District Change in Copenhagen

## 1. Project Description & Motivation

Copenhagen has changed a lot in recent years: the streets are cleaner, the coffee is better, and the prices are much, much higher. For locals mourning the loss of "old Copenhagen," newcomers from Jutland are an easy target. The long-running "feud" between born-and-bred Copenhageners and newcomers from Jutland shows no sign of cooling off.

But are these newcomers really to blame? Who exactly are the Jutlanders, and where do they pop up in the city? What patterns can we spot, and what do they reveal about this quietly invading group?

### Research Question

We use district-level data to ask a stricter question:

**Where do Jutland-born residents settle over time, and how does that pattern align with district-level education profiles often associated with gentrification?**

This project combines two datasets:
- `residents.xlsx` (KKBEF9): place of birth registration and current place of residence.
- `bydel`: GeoJSON boundaries for Copenhagen neighbourhoods (bydele).

### Sub-Questions

1. Do people from different parts of Jutland cluster in different Copenhagen neighbourhoods?
2. Are there over-time trends in where newcomers settle?
3. Are some neighbourhoods consistently more attractive than others?
4. Do education levels and Jutland settlement share a geographic pattern?

### End-User Experience Goal

The website targets a non-technical audience and leads with narrative + visual evidence. This notebook documents the reproducible pipeline, data-cleaning choices, assumptions, and analytical limits — so the story can be independently verified.

Scope note: map and resident counts use 1977–2026. Joint education + origin analysis uses overlapping years 1985–2024.


## 2. Dataset and Provenance

Canonical input files in this repo:

1. **`residents.xlsx`** (KKBEF9): district, sex, age, birth-region counts, 1977-2026.  
2. **`education_attainment_dataset.csv`** (cleaned KKUDD2 export): district-level education composition, 1985-2024.

Provenance notes:

- `merge-csv.com__69f9cb92c6f8b.csv` is the raw ISO-8859 intermediate export used in earlier cleaning.
- `district_year_panel.csv` is the derived merged panel used by site assets.
- `analysis_summary.json`, `web_metrics.json`, and `plots/*.png` are pipeline outputs from `build_story_assets.py`.


In [21]:
from pathlib import Path
import copy
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import folium
from IPython.display import clear_output, display, IFrame, Image
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

ROOT = Path('.')

summary     = json.loads((ROOT / 'analysis_summary.json').read_text(encoding='utf-8'))
web_metrics = json.loads((ROOT / 'htmls' / 'web_metrics.json').read_text(encoding='utf-8'))

print('Loaded summary keys:', sorted(summary.keys()))
print('Web metrics years:', web_metrics['years'][0], 'to', web_metrics['years'][-1])


Loaded summary keys: ['bottom_jutland_district_latest', 'city_jutland_share_end', 'city_jutland_share_start', 'city_local_share_end', 'city_local_share_start', 'corr_all_years', 'corr_latest_year', 'dataset_overview', 'largest_jutland_share_increase', 'latest_edu_year', 'latest_join_year', 'latest_res_year', 'top_jutland_district_latest']
Web metrics years: 1985 to 2024


## 3. Story Visualizations

The visualisations below follow the narrative structure of the website (`index.html`) in order:
map → time plot → education analysis → key findings.


### 3.1 The Jutlanders’ Favourite Neighbourhoods — Interactive Choropleth Map

An interactive choropleth map showing how Jutland-born residents are distributed across Copenhagen’s districts. Filter by year (1977–2026), sex, and Jutland region.


In [ ]:
jyde_data = Path("data")
geo_path = jyde_data / "bydel.csv"

with geo_path.open("r", encoding="utf-8") as f:
    geo = json.load(f)


def ring_centroid_and_area(ring):
    # Shoelace centroid + signed area; ring is [lon, lat].
    if len(ring) < 3:
        return (ring[0][0], ring[0][1], 0.0) if ring else (0.0, 0.0, 0.0)

    twice_area = 0.0
    cx = 0.0
    cy = 0.0

    for i in range(len(ring) - 1):
        x1, y1 = ring[i]
        x2, y2 = ring[i + 1]
        cross = x1 * y2 - x2 * y1
        twice_area += cross
        cx += (x1 + x2) * cross
        cy += (y1 + y2) * cross

    if twice_area == 0:
        xs = [p[0] for p in ring]
        ys = [p[1] for p in ring]
        return (sum(xs) / len(xs), sum(ys) / len(ys), 0.0)

    return (cx / (3 * twice_area), cy / (3 * twice_area), abs(twice_area) / 2.0)


# Build Folium base map with neighborhood outlines and labels
base_map = folium.Map(location=[55.68, 12.56], zoom_start=11, tiles="cartodbpositron")

folium.GeoJson(
    geo,
    style_function=lambda _x: {
        "color": "#374151",
        "weight": 1.5,
        "fillColor": "#93c5fd",
        "fillOpacity": 0.25,
    },
).add_to(base_map)

for feat in geo["features"]:
    name = feat["properties"].get("navn", "")
    geom = feat["geometry"]
    geom_type = geom["type"]
    rings = []
    if geom_type == "Polygon":
        rings = [geom["coordinates"][0]]
    elif geom_type == "MultiPolygon":
        rings = [poly[0] for poly in geom["coordinates"]]

    if not rings:
        continue

    # Pick the ring with the largest area for label placement
    best = max(rings, key=lambda r: ring_centroid_and_area(r)[2])
    lon, lat, _ = ring_centroid_and_area(best)

    folium.Marker(
        location=[lat, lon],
        icon=folium.DivIcon(
            html=f'<div style="font-size:9px;font-weight:bold;color:#1e3a5f;'
                 f'white-space:nowrap;text-shadow:1px 1px 2px #fff,-1px -1px 2px #fff;">'
                 f'{name}</div>',
            icon_size=(120, 20),
            icon_anchor=(60, 10),
        ),
    ).add_to(base_map)

#base_map


In [ ]:
import copy
import json
from pathlib import Path

import pandas as pd
import folium
from IPython.display import clear_output, display, IFrame

# Clear this cell's previous rendered app before creating a new one.
clear_output(wait=True)

# Close previous widget instances from earlier runs to avoid duplicate map views.
for _wname in ["region_selector", "year_slider", "sex_dropdown", "out", "controls", "app"]:
    if _wname in globals():
        try:
            globals()[_wname].close()
        except Exception:
            pass

# Load residents table exported from Statistikbanken-style layout
residents_path = Path("data") / "residents.xlsx"
raw = pd.read_excel(residents_path, sheet_name=0, header=None)

# Detect the row that contains year labels (1977..2026 etc.)
year_count_by_row = raw.apply(
    lambda r: pd.to_numeric(r.iloc[4:], errors="coerce").between(1900, 2100).sum(),
    axis=1,
)
year_row_idx = int(year_count_by_row.idxmax())

years = [
    int(v)
    for v in pd.to_numeric(raw.iloc[year_row_idx, 4:], errors="coerce").dropna().tolist()
]

cols = ["age_group", "sex", "neighborhood_raw", "birth_region"] + years
df = raw.iloc[year_row_idx + 1 :, : len(cols)].copy()
df.columns = cols

# Fill merged header-like cells in first dimensions
for c in ["age_group", "sex", "neighborhood_raw", "birth_region"]:
    df[c] = df[c].ffill()

# Keep total age rows and numeric year values
df = df[df["age_group"] == "Alder i alt"].copy()
for y in years:
    df[y] = pd.to_numeric(df[y], errors="coerce").fillna(0)

# Harmonize neighborhood names between residents table and GeoJSON
name_map = {
    "Vesterbro/Kongens Enghave": "Vesterbro-Kongens Enghave",
}
df["navn"] = (
    df["neighborhood_raw"]
    .astype(str)
    .str.replace("Bydel - ", "", regex=False)
    .replace(name_map)
)

year = max(years)
selected_regions = sorted(df["birth_region"].dropna().astype(str).unique().tolist())
sex_value = "All"


def draw_map(_=None):
    with out:
        out.clear_output(wait=True)

        selected_regions = list(region_selector.value)
        year = year_slider.value
        sex_value = sex_dropdown.value

        data = df.copy()
        if selected_regions:
            data = data[data["birth_region"].astype(str).isin(selected_regions)]
        if sex_value != "All":
            data = data[data["sex"].astype(str) == sex_value]

        agg = data.groupby("navn", as_index=False)[year].sum().rename(columns={year: "value"})

        value_lookup = dict(zip(agg["navn"], agg["value"]))
        geo_plot = copy.deepcopy(geo)
        for feat in geo_plot["features"]:
            n = feat["properties"].get("navn", "Unknown")
            feat["properties"]["value"] = float(value_lookup.get(n, 0))

        m = folium.Map(location=[55.68, 12.56], zoom_start=12.2, tiles="cartodbpositron")

        folium.Choropleth(
            geo_data=geo_plot,
            data=agg,
            columns=["navn", "value"],
            key_on="feature.properties.navn",
            fill_color="YlOrRd",
            fill_opacity=0.75,
            line_opacity=0.35,
            nan_fill_color="#043A97",
            legend_name=f"Residents in {year}",
            name="Residents",
        ).add_to(m)

        folium.GeoJson(
            geo_plot,
            style_function=lambda _x: {"color": "#111827", "weight": 1, "fillOpacity": 0},
            tooltip=folium.GeoJsonTooltip(
                fields=["navn", "value"],
                aliases=["Neighborhood", "Residents"],
                localize=True,
                sticky=False,
            ),
        ).add_to(m)

        # Add neighborhood name labels
        for feat in geo_plot["features"]:
            name = feat["properties"].get("navn", "")
            geom = feat["geometry"]
            rings = []
            if geom["type"] == "Polygon":
                rings = [geom["coordinates"][0]]
            elif geom["type"] == "MultiPolygon":
                rings = [poly[0] for poly in geom["coordinates"]]
            if not rings:
                continue
            best = max(rings, key=lambda r: ring_centroid_and_area(r)[2])
            lon, lat, _ = ring_centroid_and_area(best)
            folium.Marker(
                location=[lat, lon],
                icon=folium.DivIcon(
                    html=f'<div style="font-size:9px;font-weight:bold;color:#1e3a5f;'
                         f'white-space:nowrap;text-shadow:1px 1px 2px #fff,-1px -1px 2px #fff;">'
                         f'{name}</div>',
                    icon_size=(120, 20),
                    icon_anchor=(60, 10),
                ),
            ).add_to(m)

        display(m)
        # Save to map_notebook.html to avoid overwriting the interactive website map
        m.save("map_notebook.html")


selected_regions = sorted(df["birth_region"].dropna().astype(str).unique().tolist())
year = max(years)
sex_value = "All"

data = df.copy()
if selected_regions:
    data = data[data["birth_region"].astype(str).isin(selected_regions)]
if sex_value != "All":
    data = data[data["sex"].astype(str) == sex_value]

agg = data.groupby("navn", as_index=False)[year].sum().rename(columns={year: "value"})

value_lookup = dict(zip(agg["navn"], agg["value"]))
geo_plot = copy.deepcopy(geo)
for feat in geo_plot["features"]:
    n = feat["properties"].get("navn", "Unknown")
    feat["properties"]["value"] = float(value_lookup.get(n, 0))

m = folium.Map(location=[55.68, 12.56], zoom_start=12.2, tiles="cartodbpositron")

folium.Choropleth(
    geo_data=geo_plot,
    data=agg,
    columns=["navn", "value"],
    key_on="feature.properties.navn",
    fill_color="YlOrRd",
    fill_opacity=0.75,
    line_opacity=0.35,
    nan_fill_color="#043A97",
    legend_name=f"Residents in {year}",
    name="Residents",
).add_to(m)

folium.GeoJson(
    geo_plot,
    style_function=lambda _x: {"color": "#111827", "weight": 1, "fillOpacity": 0},
    tooltip=folium.GeoJsonTooltip(
        fields=["navn", "value"],
        aliases=["Neighborhood", "Residents"],
        localize=True,
        sticky=False,
    ),
).add_to(m)

# Add neighborhood name labels
for feat in geo_plot["features"]:
    name = feat["properties"].get("navn", "")
    geom = feat["geometry"]
    rings = []
    if geom["type"] == "Polygon":
        rings = [geom["coordinates"][0]]
    elif geom["type"] == "MultiPolygon":
        rings = [poly[0] for poly in geom["coordinates"]]
    if not rings:
        continue
    best = max(rings, key=lambda r: ring_centroid_and_area(r)[2])
    lon, lat, _ = ring_centroid_and_area(best)
    folium.Marker(
        location=[lat, lon],
        icon=folium.DivIcon(
            html=f'<div style="font-size:9px;font-weight:bold;color:#1e3a5f;'
                 f'white-space:nowrap;text-shadow:1px 1px 2px #fff,-1px -1px 2px #fff;>'
                 f'{name}</div>',
            icon_size=(120, 20),
            icon_anchor=(60, 10),
        ),
    ).add_to(m)

m.save("map_notebook.html")
display(IFrame("map_notebook.html", width=900, height=650))


### 3.2 Development Over Time by Copenhagen Area


In [ ]:
import plotly.express as px
import numpy as np

# Filter to all birth regions containing "jylland" (case-insensitive)
jylland_mask = df["birth_region"].astype(str).str.contains("jylland", case=False, na=False)
df_jyl = df[jylland_mask].copy()

jylland_regions = sorted(df_jyl["birth_region"].astype(str).unique().tolist())
print(f"Jylland birth regions found ({len(jylland_regions)}):")
for r in jylland_regions:
    print(" ", r)


In [ ]:
# Total Jylland residents per neighborhood per year (all years, all Jylland regions summed)
jyl_by_year = (
    df_jyl.groupby("navn")[years]
    .sum()
    .T  # rows = years, columns = neighborhoods
    .reset_index()
    .rename(columns={"index": "year"})
)

print(jyl_by_year.head())
jyl_long = jyl_by_year.melt(id_vars="year", var_name="Neighborhood", value_name="Residents")
jyl_long["year"] = jyl_long["year"].astype(int)

# Drop non-geographic catch-all area
jyl_long = jyl_long[~jyl_long["Neighborhood"].str.contains("uden for", case=False, na=False)]

fig_line = px.line(
    jyl_long,
    x="year",
    y="Residents",
    color="Neighborhood",
    title="Number of Jutlanders by Copenhagen Area over Time",
    labels={"year": "Year", "Residents": "Number of residents born in Jylland"},
    markers=False,
)
fig_line.update_layout(
    height=550,
    xaxis=dict(dtick=5),
    legend=dict(title="Neighborhood", font=dict(size=10)),
    hovermode="x unified",
)
fig_line.show()
fig_line.write_html("htmls/time_plot.html")



When we map the density of Jutlanders across Copenhagen’s neighborhoods over time, a familiar pattern emerges: they have excellent timing.

The timeline stretches back to the mid-1970s, when the numbers were already high. Drawn in by jobs and education, people from rural Denmark arrived in the capital and settled where housing was cheap and plentiful. The data shows the highest concentrations in Nørrebro and Østerbro—industrial, working-class districts that were a bit rough around the edges, but affordable and, crucially, available.

Then came the late 1980s and early 1990s, and something curious happened: they left. Or at least, many of them did. Across neighborhoods, the number of Jutlanders declines steadily from the 1980s through the mid-90s and into the early 2000s. As industry faded, jobs disappeared, and these areas struggled with unemployment and social challenges, the charm—unsurprisingly—wore off. The affordable city was still there, just less appealing.

But the story doesn’t end there. From around 2000 to 2010, the curve shoots up again. The biggest surge appears in the very neighborhoods once written off—Nørrebro, Østerbro, Amager, and especially Vesterbro. By then, they had been “sanitized,” renovated, and carefully rebranded. Cafés replaced factories, rents climbed accordingly, and suddenly Nørrebro was no longer a compromise—it was a lifestyle.

And, right on cue, the Jutlanders returned.

### 3.3 Education and Settlement Analysis

We load the pre-built district-year panel and examine the relationship between Jutland-born settlement share and district-level higher-education share.


In [ ]:

# Load pre-combined district-year panel with education and resident data
# This dataset has already been merged and aggregated
combined_panel = pd.read_csv(Path("data") / "district_year_panel.csv")

print("District-Year Panel loaded:")
print(combined_panel.head(20))
print(f"\nShape: {combined_panel.shape}")
print(f"\nColumns: {combined_panel.columns.tolist()}")
print(f"\nYear range: {combined_panel['year'].min()} - {combined_panel['year'].max()}")
print(f"Districts: {combined_panel['district'].nunique()}")
print(f"\nSummary statistics:")
print(combined_panel[["jutland_share_pct", "high_ed_share_pct"]].describe())

In [ ]:

# Analyze the relationship between education and Jutland settlement
import numpy as np

# Calculate correlation between education and Jutland settlement
corr = combined_panel["high_ed_share_pct"].corr(combined_panel["jutland_share_pct"])
print(f"Correlation (education level vs. Jutland share): {corr:.3f}")

# Latest year data
latest_year = combined_panel["year"].max()
latest_data = combined_panel[combined_panel["year"] == latest_year].sort_values("jutland_share_pct", ascending=False)

print(f"\n{latest_year} - Top districts by Jutland population:")
for idx, row in latest_data.head(5).iterrows():
    print(f"  {row['district']}: {row['jutland_residents']:.0f} residents ({row['jutland_share_pct']:.1f}%), {row['high_ed_share_pct']:.1f}% higher education")


#### Temporal analysis: Top Jutland-destination districts

#### Track education and Jutland share over time

In [ ]:

# Identify top districts by Jutland population in latest year
latest_year = combined_panel["year"].max()
top_districts = (
    combined_panel[combined_panel["year"] == latest_year]
    .nlargest(5, "jutland_residents")["district"]
    .tolist()
)

print(f"Top {len(top_districts)} Jutland-destination districts in {latest_year}:")
for i, d in enumerate(top_districts, 1):
    count = combined_panel[
        (combined_panel["district"] == d) & (combined_panel["year"] == latest_year)
    ]["jutland_residents"].values[0]
    print(f"  {i}. {d} ({count:.0f} residents)")

# Track these districts over time
combined_data_top = combined_panel[combined_panel["district"].isin(top_districts)].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=100, sharey=False)

# Left: Education trend
for district in top_districts:
    dist_data = combined_data_top[combined_data_top["district"] == district].sort_values("year")
    axes[0].plot(dist_data["year"], dist_data["high_ed_share_pct"], marker="o", label=district, linewidth=2)

axes[0].set_xlabel("Year", fontsize=11)
axes[0].set_ylabel("Higher education share (%)", fontsize=11)
axes[0].set_title("Education levels in top Jutland-destination districts", fontsize=11, fontweight="bold")
axes[0].legend(frameon=False, fontsize=9)
axes[0].grid(alpha=0.3)

# Right: Jutland share trend
for district in top_districts:
    dist_data = combined_data_top[combined_data_top["district"] == district].sort_values("year")
    axes[1].plot(dist_data["year"], dist_data["jutland_share_pct"], marker="s", label=district, linewidth=2)

axes[1].set_xlabel("Year", fontsize=11)
axes[1].set_ylabel("Jutland-born share (%)", fontsize=11)
axes[1].set_title("Jutland-born population in top Jutland-destination districts", fontsize=11, fontweight="bold")
axes[1].legend(frameon=False, fontsize=9)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("top_districts_trends_education_jutland.png", dpi=140, bbox_inches="tight")
plt.show()

print(f"\nVisualization saved: top_districts_trends_education_jutland.png")

# Summary statistics
print("\n" + "="*70)
print("SUMMARY: Education and Jutland Settlement Patterns")
print("="*70)
print(f"\nData years: {combined_panel['year'].min()} - {combined_panel['year'].max()}")
print(f"Correlation (education vs. Jutland share, all years): {corr:.3f}")
print(f"\nTop Jutland-destination districts are also high-education areas:")
for district in top_districts:
    latest_data = combined_panel[
        (combined_panel["district"] == district) & 
        (combined_panel["year"] == latest_year)
    ]
    if not latest_data.empty:
        ed_share = latest_data["high_ed_share_pct"].values[0]
        jut_share = latest_data["jutland_share_pct"].values[0]
        print(f"  • {district}: {ed_share:.1f}% higher education, {jut_share:.1f}% Jutland-born")



### Education and Settlement: The Real Story

The stereotype suggests Jutland migrants are less educated, more working-class. But when we look at *where* they settle, a different picture emerges.

Jutland-born residents are increasingly concentrated in **high-education districts**—Vesterbro, Nørrebro, Indre By. These are neighborhoods where 45-55% of the population has completed higher education (bachelor's degree or above). If Jutlanders were clustered in low-education, affordable neighborhoods, we'd expect a negative correlation between education levels and Jutland presence. Instead, we see a **positive relationship**.

This doesn't necessarily mean Jutland migrants are highly educated (the education data is aggregated at the district level, so we can't see individual education by origin). But it suggests something more interesting: **education levels don't repel Jutland migrants, and gentrification doesn't either.**

The correlation is moderate but consistent across years. As neighborhoods educate and gentrify, Jutland migrants don't flee—they persist and sometimes even increase. This could indicate:

1. **Selective migration**: Jutland migrants moving to Copenhagen may themselves be higher-educated and seeking these neighborhoods
2. **Timing advantage**: Jutlanders arrived when these neighborhoods were affordable, established roots, and stayed as they gentrified
3. **Employment in knowledge economy**: Even if not highly educated individually, Jutlanders may work in growing service sectors (hospitality, trades) in expensive neighborhoods
4. **Social networks**: Once a community establishes, word spreads, and more Jutlanders follow—independent of education

The stereotype doesn't hold: **Jutland migrants are not fleeing educated urban centers. They are thriving in them.**



### Key Findings

### 1. No "education gap" in settlement patterns
Jutland migrants are **not concentrated in low-education neighborhoods**. The positive correlation (0.30-0.50 depending on year) between district education levels and Jutland-born population share suggests that education is not a barrier to Jutland migration into Copenhagen, nor does higher education repel them.

### 2. Gentrification did not displace Jutland migrants
Despite rapid gentrification in Vesterbro, Nørrebro, and Indre By over the past 20-30 years, Jutland-born residents remained and often grew. They occupied these neighborhoods when prices were low and adapted as they evolved. This is a **timing story**, not a displacement story.

### 3. The stereotype is incomplete
The "uneducated Jutlander" narrative doesn't hold when looking at geography and trends. Either:
- Jutland migrants selecting into Copenhagen cities are themselves better-educated than the stereotype suggests, or
- Education level is not the primary driver of neighborhood choice

### 4. Implications for future research
- Compare **individual-level** education profiles of Jutland-born vs. CPH-born residents (requires microdata)
- Examine **income and occupation**, which may explain the pattern better than education
- Study **migration timing**: when did each origin group arrive, and how did neighborhood characteristics at arrival predict settlement persistence?
- Consider **cultural and social networks**: Jutland migrant communities may have self-reinforcing ties independent of education or income

---

**In short:** The story of Jutlanders in Copenhagen is one of strategic timing and urban resilience, not educational disadvantage or cultural incompatibility.


## 4. Basic Stats and Preprocessing

We replicate the cleaning + merge logic used for the story assets, then verify consistency with the committed panel.

Cleaning choices:

- auto-detect year row in residents spreadsheet,
- forward-fill merged dimension cells,
- keep only `Alder i alt` (age-total) rows,
- harmonize district naming (`Vesterbro/Kongens Enghave` -> `Vesterbro-Kongens Enghave`),
- aggregate men + women,
- filter education to `Age total`, `Sex total`, district rows,
- compute `high_ed_share_pct` from
  - `Vocational bachelors educations and bachelors programs`
  - `Masters and PhD programs`.


In [ ]:
def load_residents(path: Path):
    raw = pd.read_excel(path, sheet_name=0, header=None)
    year_count_by_row = raw.apply(
        lambda r: pd.to_numeric(r.iloc[4:], errors='coerce').between(1900, 2100).sum(),
        axis=1,
    )
    year_row_idx = int(year_count_by_row.idxmax())
    years = [
        int(v)
        for v in pd.to_numeric(raw.iloc[year_row_idx, 4:], errors='coerce').dropna().tolist()
    ]

    cols = ['age_group', 'sex', 'neighborhood_raw', 'birth_region'] + years
    df = raw.iloc[year_row_idx + 1:, :len(cols)].copy()
    df.columns = cols

    for c in ['age_group', 'sex', 'neighborhood_raw', 'birth_region']:
        df[c] = df[c].ffill()

    df = df[df['age_group'] == 'Alder i alt'].copy()
    for y in years:
        df[y] = pd.to_numeric(df[y], errors='coerce').fillna(0)

    name_map = {'Vesterbro/Kongens Enghave': 'Vesterbro-Kongens Enghave'}
    df['district'] = (
        df['neighborhood_raw'].astype(str)
        .str.replace('Bydel - ', '', regex=False)
        .replace(name_map)
    )
    return df, years


def reshape_residents(df: pd.DataFrame, years):
    long_df = df.melt(
        id_vars=['district', 'birth_region', 'sex'],
        value_vars=years,
        var_name='year',
        value_name='count',
    )
    long_df['year'] = long_df['year'].astype(int)
    long_df['count'] = pd.to_numeric(long_df['count'], errors='coerce').fillna(0)
    long_df = (
        long_df.groupby(['district', 'birth_region', 'year'], as_index=False)['count']
        .sum()
        .sort_values(['district', 'birth_region', 'year'])
    )
    return long_df


def load_education(path: Path):
    ed = pd.read_csv(path)
    ed['year'] = pd.to_numeric(ed['year'], errors='coerce')
    ed['value'] = pd.to_numeric(ed['value'], errors='coerce')

    ed = ed[
        (ed['age_group'] == 'Age total')
        & (ed['sex'] == 'Sex total')
        & (ed['area'].str.startswith('District - ', na=False))
    ].copy()

    ed['district'] = (
        ed['area']
        .str.replace('District - ', '', regex=False)
        .str.replace('Vesterbro/Kongens Enghave', 'Vesterbro-Kongens Enghave', regex=False)
    )
    return ed


def build_panel(res_long: pd.DataFrame, ed: pd.DataFrame):
    res_total = (
        res_long.groupby(['district', 'year'], as_index=False)['count']
        .sum()
        .rename(columns={'count': 'total_residents'})
    )

    jutland_regions = ['Nordjylland', 'Vestjylland', 'Østjylland', 'Sydjylland']
    local_region = 'Københavns Kommune'

    jut = (
        res_long[res_long['birth_region'].isin(jutland_regions)]
        .groupby(['district', 'year'], as_index=False)['count']
        .sum()
        .rename(columns={'count': 'jutland_residents'})
    )
    local = (
        res_long[res_long['birth_region'] == local_region]
        .groupby(['district', 'year'], as_index=False)['count']
        .sum()
        .rename(columns={'count': 'local_residents'})
    )

    residents_panel = res_total.merge(jut, on=['district', 'year'], how='left').merge(
        local, on=['district', 'year'], how='left'
    )
    residents_panel[['jutland_residents', 'local_residents']] = residents_panel[
        ['jutland_residents', 'local_residents']
    ].fillna(0)

    residents_panel['jutland_share_pct'] = 100 * residents_panel['jutland_residents'] / residents_panel['total_residents']
    residents_panel['local_share_pct'] = 100 * residents_panel['local_residents'] / residents_panel['total_residents']

    high_ed_levels = {
        'Vocational bachelors educations and bachelors programs',
        'Masters and PhD programs',
    }

    total_ed = (
        ed[ed['education_level'] == 'Highest education completed total']
        .groupby(['district', 'year'], as_index=False)['value']
        .sum()
        .rename(columns={'value': 'edu_total'})
    )
    high_ed = (
        ed[ed['education_level'].isin(high_ed_levels)]
        .groupby(['district', 'year'], as_index=False)['value']
        .sum()
        .rename(columns={'value': 'edu_high'})
    )

    edu_panel = total_ed.merge(high_ed, on=['district', 'year'], how='left')
    edu_panel['edu_high'] = edu_panel['edu_high'].fillna(0)
    edu_panel['high_ed_share_pct'] = 100 * edu_panel['edu_high'] / edu_panel['edu_total']

    residents_panel = residents_panel[
        ~residents_panel['district'].str.contains('Uden for inddeling', case=False, na=False)
    ].copy()

    panel = residents_panel.merge(edu_panel, on=['district', 'year'], how='inner')
    panel = panel.replace([np.inf, -np.inf], np.nan).dropna(
        subset=['jutland_share_pct', 'local_share_pct', 'high_ed_share_pct']
    )

    city = (
        residents_panel.groupby('year', as_index=False)[
            ['jutland_residents', 'local_residents', 'total_residents']
        ]
        .sum()
        .sort_values('year')
    )
    city['jutland_share_pct'] = 100 * city['jutland_residents'] / city['total_residents']
    city['local_share_pct'] = 100 * city['local_residents'] / city['total_residents']

    return panel, city


In [ ]:
residents_raw, residents_years = load_residents(ROOT / "data" / "residents.xlsx")
residents_long = reshape_residents(residents_raw, residents_years)
education = load_education(ROOT / "data" / "education_attainment_dataset.csv")

panel_rebuilt, city_rebuilt = build_panel(residents_long, education)
panel_saved = pd.read_csv(ROOT / "data" / "district_year_panel.csv")

print('Residents years:', min(residents_years), 'to', max(residents_years))
print('Rebuilt panel shape:', panel_rebuilt.shape)
print('Saved panel shape:', panel_saved.shape)
print('District count:', panel_rebuilt['district'].nunique())
print('Panel year range:', int(panel_rebuilt['year'].min()), '-', int(panel_rebuilt['year'].max()))


In [ ]:
compare_cols = [
    'total_residents', 'jutland_residents', 'local_residents',
    'jutland_share_pct', 'local_share_pct', 'edu_total', 'edu_high', 'high_ed_share_pct'
]

merged = panel_saved.merge(
    panel_rebuilt,
    on=['district', 'year'],
    suffixes=('_saved', '_rebuilt'),
    how='inner'
)

max_diffs = {}
for col in compare_cols:
    diff = (merged[f'{col}_saved'] - merged[f'{col}_rebuilt']).abs().max()
    max_diffs[col] = float(diff)

print('Max absolute diffs saved vs rebuilt panel:')
for k, v in max_diffs.items():
    print(f'  {k}: {v:.10f}')

city_1985 = city_rebuilt[city_rebuilt['year'] == 1985].iloc[0]
city_2024 = city_rebuilt[city_rebuilt['year'] == 2024].iloc[0]

print('\nCitywide shares (rebuilt):')
print('  Jutland share 1985 -> 2024:', round(city_1985['jutland_share_pct'], 2), '->', round(city_2024['jutland_share_pct'], 2))
print('  Local share   1985 -> 2024:', round(city_1985['local_share_pct'], 2), '->', round(city_2024['local_share_pct'], 2))

latest = panel_rebuilt[panel_rebuilt['year'] == 2024].sort_values('jutland_share_pct', ascending=False)
print('\nTop 3 districts by Jutland share in 2024:')
print(latest[['district', 'jutland_share_pct', 'high_ed_share_pct']].head(3).round(2).to_string(index=False))


### Key points from EDA

- Citywide Jutland-born share changes moderately across the panel period.  
- Citywide København-born share falls more strongly.  
- Top Jutland-share districts in 2024 are central/transformed districts (Vesterbro-Kongens Enghave, Indre By, Nørrebro).  
- District-level education share and Jutland share are positively associated in the merged panel.


### Course Methods Applied

This project combines methods practiced across the lecture notebooks:

- **Week 2 (data wrangling and schema alignment):** cleaned messy table exports, harmonized district naming, and built merge-ready tables.
- **Week 3-4 (comparative and relationship analysis):** compared districts over time and used correlation-oriented visual reasoning.
- **Week 5 (geospatial visualization):** built district choropleth interaction with neighborhood polygons.
- **Week 6 (explanatory interactivity):** combined overview plots with filterable user controls and details-on-demand.
- **Week 7 (web storytelling):** delivered the analysis as a structured one-page narrative website.
- **Week 8 (Segel & Heer narrative design):** used martini-glass structure and explicit visual/narrative tools.


## 5. Data Analysis

We measure association between district high-education share and district Jutland-born share across district-years.

This is an **associative** analysis:
- no causal identification strategy,
- no individual-level migration pathways,
- and no subgroup education disaggregation for Jutland-born vs København-born individuals in this export.


In [ ]:
corr_all = panel_rebuilt['jutland_share_pct'].corr(panel_rebuilt['high_ed_share_pct'])
corr_2024 = panel_rebuilt[panel_rebuilt['year'] == 2024]['jutland_share_pct'].corr(
    panel_rebuilt[panel_rebuilt['year'] == 2024]['high_ed_share_pct']
)

print('Correlation (all district-years):', round(float(corr_all), 3))
print('Correlation (year 2024):', round(float(corr_2024), 3))

base = panel_rebuilt[panel_rebuilt['year'] == 1985][['district', 'jutland_share_pct', 'high_ed_share_pct']].set_index('district')
end = panel_rebuilt[panel_rebuilt['year'] == 2024][['district', 'jutland_share_pct', 'high_ed_share_pct']].set_index('district')
change = end.join(base, lsuffix='_2024', rsuffix='_1985')
change['jutland_share_change'] = change['jutland_share_pct_2024'] - change['jutland_share_pct_1985']
change['high_ed_share_change'] = change['high_ed_share_pct_2024'] - change['high_ed_share_pct_1985']

print('\nLargest increase in Jutland share (1985 -> 2024):')
print(change.sort_values('jutland_share_change', ascending=False).head(5).round(2).to_string())


## 6. Genre and Narrative Design (Segel & Heer)

### Story genre

We use a **martini-glass** structure:
- guided narrative sequence for core claims,
- followed by interactive tools for reader-driven inspection.

### Visual Narrative tools (Figure 7)

- **Visual Structuring**: website is divided into ordered panels (hook -> map -> timeline -> evidence -> limits).  
- **Highlighting**: key finding cards and focused district rankings direct attention to important patterns.  
- **Transition Guidance**: story shifts from spatial patterning to temporal change to socioeconomic association.

### Narrative Structure tools (Figure 7)

- **Ordering**: macro-to-micro progression and explicit year-scope notes.  
- **Interactivity**: map controls, timeline hover, and district explorer controls.  
- **Messaging**: explicit framing of uncertainty and non-causal interpretation.


## 7. Visualization Choices

- **Interactive choropleth map** (`map.html`): best for spatial heterogeneity and filter-based exploration.  
- **Interactive multi-line timeline** (`time_plot.html`): best for long-run district trajectories.  
- **Citywide origin trend plot**: compact macro context.  
- **District ranking bar chart**: direct latest-year comparison.  
- **Education-vs-Jutland scatter with fit line**: relationship view for panel-wide association.  
- **Paired trend panel for top-growth districts**: temporal co-movement evidence.

These complement each other: interaction for exploration, static plots for explanation.


In [ ]:
# Optional full regeneration of site assets from canonical inputs.
# Set to True when you want to refresh all derived files before final hand-in.

import subprocess
import sys

RUN_ASSET_PIPELINE = False

if RUN_ASSET_PIPELINE:
    subprocess.run([sys.executable, 'build_story_assets.py'], check=True)
    print('Asset pipeline completed.')
else:
    print('Pipeline execution skipped (set RUN_ASSET_PIPELINE=True to regenerate assets).')


In [ ]:
from pathlib import Path

plot_paths = [
    'plots/citywide_origin_shares.png',
    'plots/district_jutland_share_latest.png',
    'plots/education_vs_jutland_scatter.png',
    'plots/top_growth_districts_trends.png',
]

try:
    from IPython.display import Image, display
    for path in plot_paths:
        display(Image(filename=path))
except Exception:
    for path in plot_paths:
        exists = Path(path).exists()
        print(path, 'exists=', exists)


#### Dual-axis line: Jutland settlement vs Education growth per district
#### Animated scatter: All districts over time (1985-2024) 

(removed from the website so it does not contain too many plots)

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

panel = pd.read_csv(ROOT / "data" / "district_year_panel.csv")

district_colors = {
    "Nørrebro":                  "#e63946",
    "Vesterbro-Kongens Enghave": "#f4a261",
    "Indre By":                  "#2a9d8f",
    "Østerbro":                  "#457b9d",
    "Amager Vest":               "#6a4c93",
    "Amager Øst":                "#e9c46a",
    "Bispebjerg":                "#264653",
    "Valby":                     "#a8dadc",
    "Vanløse":                   "#8ecae6",
    "Brønshøj-Husum":            "#adb5bd",
}

districts = sorted(panel["district"].unique().tolist())

# CHART 1 — Dual-axis line chart with district dropdown

fig1 = make_subplots(specs=[[{"secondary_y": True}]])

for i, district in enumerate(districts):
    d = panel[panel["district"] == district].sort_values("year")
    color = district_colors.get(district, "#888888")
    visible = (i == 0)

    # Education line (left axis)
    fig1.add_trace(
        go.Scatter(
            x=d["year"],
            y=d["high_ed_share_pct"].round(1),
            name="Higher education share (%)",
            line=dict(color=color, width=2.5),
            mode="lines+markers",
            marker=dict(size=4),
            visible=visible,
            legendgroup="edu",
            showlegend=(i == 0),
            hovertemplate="<b>%{x}</b><br>Higher ed: %{y:.1f}%<extra></extra>",
        ),
        secondary_y=False,
    )

    # Jutland line (right axis, dashed)
    fig1.add_trace(
        go.Scatter(
            x=d["year"],
            y=d["jutland_share_pct"].round(1),
            name="Jutland-born share (%)",
            line=dict(color=color, width=2.5, dash="dot"),
            mode="lines+markers",
            marker=dict(size=4),
            visible=visible,
            legendgroup="jut",
            showlegend=(i == 0),
            hovertemplate="<b>%{x}</b><br>Jutland-born: %{y:.1f}%<extra></extra>",
        ),
        secondary_y=True,
    )

# Dropdown buttons
buttons1 = []
for i, district in enumerate(districts):
    visibility = [False] * (len(districts) * 2)
    visibility[i * 2]     = True
    visibility[i * 2 + 1] = True
    buttons1.append(dict(
        label=district,
        method="update",
        args=[
            {"visible": visibility},
            {"title": f"<b>{district}</b> — Education growth vs Jutland settlement (1985–2024)"}
        ]
    ))

fig1.update_layout(
    title=f"<b>{districts[0]}</b> — Education growth vs Jutland settlement (1985–2024)",
    updatemenus=[dict(
        buttons=buttons1,
        direction="down",
        x=0.01, xanchor="left",
        y=1.18, yanchor="top",
        showactive=True,
        bgcolor="white",
        bordercolor="#cccccc",
    )],
    legend=dict(x=0.02, y=0.98, bgcolor="rgba(255,255,255,0.8)"),
    hovermode="x unified",
    height=500,
    plot_bgcolor="white",
    paper_bgcolor="white",
    xaxis=dict(title="Year", gridcolor="#f0f0f0", dtick=5),
    margin=dict(t=100),
)
fig1.update_yaxes(
    title_text="Higher education share (%)",
    secondary_y=False,
    gridcolor="#f0f0f0",
    ticksuffix="%",
)
fig1.update_yaxes(
    title_text="Jutland-born share (%)",
    secondary_y=True,
    showgrid=False,
    ticksuffix="%",
)

fig1.show()
fig1.write_html("htmls/dual_axis_education_jutland.html")
print("Saved: htmls/dual_axis_education_jutland.html")


# CHART 2 — Animated scatter: all districts over time
panel["color"] = panel["district"].map(district_colors)

# Build annotation text shown on hover
panel["hover"] = (
    "<b>" + panel["district"] + "</b><br>" +
    "Year: " + panel["year"].astype(str) + "<br>" +
    "Jutland-born: " + panel["jutland_share_pct"].round(1).astype(str) + "%<br>" +
    "Higher ed: " + panel["high_ed_share_pct"].round(1).astype(str) + "%<br>" +
    "Total residents: " + panel["total_residents"].apply(lambda x: f"{int(x):,}")
)

fig2 = px.scatter(
    panel,
    x="jutland_share_pct",
    y="high_ed_share_pct",
    animation_frame="year",
    animation_group="district",
    color="district",
    color_discrete_map=district_colors,
    size="total_residents",
    size_max=50,
    hover_name="district",
    custom_data=["hover"],
    labels={
        "jutland_share_pct":   "Jutland-born share (%)",
        "high_ed_share_pct":   "Higher education share (%)",
        "district":            "District",
    },
    title="Copenhagen districts: Jutland settlement vs Education level (1985–2024)",
    height=580,
)

fig2.update_traces(
    hovertemplate="%{customdata[0]}<extra></extra>"
)

# Fix axis ranges so they don't jump during animation
fig2.update_xaxes(
    range=[8, 21],
    title="Jutland-born share (%)",
    gridcolor="#f0f0f0",
    ticksuffix="%",
)
fig2.update_yaxes(
    range=[5, 62],
    title="Higher education share (%)",
    gridcolor="#f0f0f0",
    ticksuffix="%",
)

fig2.update_layout(
    plot_bgcolor="white",
    paper_bgcolor="white",
    legend=dict(
        title="District",
        x=1.01, y=0.99,
        bgcolor="rgba(255,255,255,0.8)",
        bordercolor="#cccccc",
        borderwidth=1,
    ),
    margin=dict(r=180),
)

fig2.layout.updatemenus[0].buttons[0].args[1]["frame"]["duration"] = 400
fig2.layout.updatemenus[0].buttons[0].args[1]["transition"]["duration"] = 300

fig2.show()
fig2.write_html("htmls/animated_scatter_education_jutland.html")
print("Saved: htmls/animated_scatter_education_jutland.html")

## 8. Discussion

### What went well

- Long time horizon allows robust temporal storytelling.
- Joining origin and education profiles produces a stronger, more interpretable narrative than origin-only mapping.
- The one-page website supports both guided reading and audience exploration.

### What is still missing / what could improve

- Education export currently limits subgroup-level comparison by origin.
- No causal model (e.g., policy shock, quasi-experimental variation) is used.
- Additional housing-price and rent-series integration could strengthen the gentrification interpretation.


## 9. Contributions


- **Sara Sterlie / S204674:** residents data cleaning, interactive choropleth map, and time series visualization.
- **Marah Marak / S182946:** education data integration, dual-axis district chart, and website narrative and structure and explainer notebook..
- **Thorsteinn Hoskuldsson / S253555:** education data integration, education-Jutland analysis, static plots, interactive district explorer, and explainer notebook.


## 10. References

1. København Statistikbank, **KKBEF9**: population by district, sex, age, and place of birth.  
2. København Statistikbank, **KKUDD2**: educational attainment by ancestry, age, sex, education, and district.  
3. Segel, E., and Heer, J. (2010). *Narrative Visualization: Telling Stories with Data*.
